# 03 - Model Training, Evaluation & Explainability
Compares several models using a metric set that prioritizes Recall/PR-AUC (see `src/evaluation.py`, `src/models.py`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/stroke-prediction/blob/main/notebooks/03_model_training_and_evaluation.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/stroke-prediction/blob/main/notebooks/03_model_training_and_evaluation.ipynb)


In [1]:
# Colab/Kaggle bootstrap requires GitHub and PyPI access; data is bundled.
import os, sys, subprocess
from pathlib import Path

cwd = Path.cwd().resolve()
repo = next((p for p in (cwd, *cwd.parents)
             if (p / "src" / "config.py").is_file() and (p / "requirements.txt").is_file()), None)
if repo is None:
    if "google.colab" in sys.modules or Path("/kaggle/working").exists():
        base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
        repo = (base / "stroke-prediction").resolve()
        if not repo.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/fossbk-spec/stroke-prediction.git", str(repo)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(repo / "requirements.txt")], check=True)
    else:
        raise RuntimeError("Run this notebook from inside the stroke-prediction repository.")
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))


In [2]:
from src.data_loader import load_raw_data
from src.train import run_experiment

df = load_raw_data()
table, cv_tables = run_experiment(
    df,
    model_names=['logistic_regression', 'random_forest', 'gradient_boosting', 'stacking'],
    use_smote=True,
    output_dir=None,
)
table

[OK] ZERO DATA LEAKAGE confirmed (Train=3,270 IDs | Validation=818 IDs | Test=1,022 IDs).

--- Data distribution after patient-level split ---
split  n_samples  n_patients  n_positive  positive_rate
train       3270        3270         159       0.048624
  val        818         818          40       0.048900
 test       1022        1022          50       0.048924

=== logistic_regression (smote=True, class_weight=None) ===
  Best threshold on validation: 0.6385 (F2=0.4230)
logistic_regression @ threshold=0.638
  TP=38  FN=12  FP=174  TN=798
  recall          = 0.7600
  pr_auc          = 0.1761
  f2              = 0.4612
  f1              = 0.2901
  precision       = 0.1792
  roc_auc         = 0.8569
  specificity     = 0.8210
  balanced_accuracy= 0.7905
  accuracy        = 0.8180
  null_accuracy   = 0.9511
  brier           = 0.1715
  n_false_negative= 12.0000
  n_true_positive = 38.0000
  n_false_positive= 174.0000
  CV (train+val):
    {'recall': '0.7085 ± 0.0679', 'pr_auc': '0.1999

,model,threshold,recall,pr_auc,f2,f1,precision,roc_auc,specificity,balanced_accuracy,accuracy,null_accuracy,brier,n_false_negative,n_true_positive,n_false_positive
0,logistic_regression,0.638473,0.76,0.176108,0.461165,0.290076,0.179245,0.856934,0.820988,0.790494,0.818004,0.951076,0.171514,12.0,38.0,174.0
1,gradient_boosting,0.126561,0.70,0.209646,0.408879,0.251799,0.153509,0.837891,0.801440,0.750720,0.796477,0.951076,0.045389,15.0,35.0,193.0
2,random_forest,0.134178,0.70,0.205668,0.422705,0.265152,0.163551,0.837222,0.815844,0.757922,0.810176,0.951076,0.045073,15.0,35.0,179.0
3,stacking,0.077222,0.54,0.214546,0.395894,0.282723,0.191489,0.843786,0.882716,0.711358,0.865949,0.951076,0.042975,23.0,27.0,114.0


## Comparison table (sorted by Recall)

In [3]:
cols = ['model','threshold','recall','pr_auc','f2','precision','roc_auc','specificity','n_false_negative']
table[cols].round(4)

,model,threshold,recall,pr_auc,f2,precision,roc_auc,specificity,n_false_negative
0,logistic_regression,0.6385,0.76,0.1761,0.4612,0.1792,0.8569,0.8210,12.0
1,gradient_boosting,0.1266,0.70,0.2096,0.4089,0.1535,0.8379,0.8014,15.0
2,random_forest,0.1342,0.70,0.2057,0.4227,0.1636,0.8372,0.8158,15.0
3,stacking,0.0772,0.54,0.2145,0.3959,0.1915,0.8438,0.8827,23.0


## Clinical explainability (XAI): SHAP + Odds Ratio

In [4]:
from src.data_loader import split_features_target
from src.preprocessing import patient_level_split, make_full_pipeline
from src.models import get_model
from src import config, explainability

split = patient_level_split(df)
X_train, y_train = split_features_target(split.train)

rf_pipeline = make_full_pipeline(get_model('random_forest', class_weight='balanced'), scale=False)
rf_pipeline.fit(X_train, y_train)
values, importance = explainability.shap_summary(
    rf_pipeline, X_train, save_path='reports/figures/shap_summary_rf.png'
)
importance.head(10)

[OK] ZERO DATA LEAKAGE confirmed (Train=3,270 IDs | Validation=818 IDs | Test=1,022 IDs).


Matplotlib is building the font cache; this may take a moment.


,feature,mean_abs_shap
0,age,0.111796
1,Residence_type_Rural,0.047360
2,smoking_status_formerly smoked,0.045176
3,hypertension,0.044261
4,ever_married,0.039476
5,Residence_type_Urban,0.036110
6,heart_disease,0.026447
7,gender_Male,0.023279
8,smoking_status_never smoked,0.023233
9,gender_Female,0.021937


In [5]:
lr_pipeline = make_full_pipeline(get_model('logistic_regression'))
lr_pipeline.fit(X_train, y_train)
explainability.logistic_odds_ratios(
    lr_pipeline, X_train, y_train,
    groups=split.train[config.ID_COLUMN], n_boot=50,
).head(10)

,feature,coefficient,odds_ratio,direction,or_ci_lower,or_ci_upper,significant
0,age,1.937082,6.938473,increases risk,5.827458e+00,8.261306e+00,True
1,hypertension,0.538463,1.713372,increases risk,1.278777e+00,2.295665e+00,True
2,work_type_Self-employed,-0.500035,0.606509,decreases risk,9.859677e-305,1.014232e+304,False
3,work_type_children,0.433289,1.542322,increases risk,9.859677e-305,1.014232e+304,False
4,smoking_status_never smoked,-0.421211,0.656251,decreases risk,9.859677e-305,1.014232e+304,False
5,Residence_type_Rural,-0.419897,0.657115,decreases risk,9.859677e-305,1.014232e+304,False
6,gender_Male,-0.356752,0.699946,decreases risk,9.859677e-305,1.014232e+304,False
7,heart_disease,0.342934,1.409076,increases risk,9.332623e-01,2.127478e+00,False
8,work_type_Govt_job,-0.295847,0.743901,decreases risk,9.859677e-305,1.014232e+304,False
9,work_type_Never_worked,-0.257296,0.773139,decreases risk,9.859677e-305,1.014232e+304,False
